# 06 · Baseline NEWS2 — la tasa de alertas de referencia

**Pregunta que responde:** ¿cuánto alerta NEWS2 **en esta cohorte**, medido con la misma
regla que el sistema aplica en producción? Es el número contra el cual se compara el
modelo en `SCRUM-58`, y sin él esa comparativa no significa nada.

| | |
|---|---|
| **Issue** | `SCRUM-80` |
| **Notebook previo** | `05_eda_y_prevalencia.ipynb` |
| **Entrada** | `data/processed/mimic-iv-ed-demo-2.2/observaciones_particionadas.parquet` y `episodios.parquet` |
| **Dataset** | MIMIC-IV-ED **demo v2.2** (subconjunto abierto, licencia ODbL). `SCRUM-48` sigue esperando la credencial de PhysioNet. |
| **Salida** | `data/interim/tasa_alertas_referencia.parquet` y figuras en `data/interim/figuras/` |
| **Semilla** | ninguna: **este notebook no sortea nada.** Todo lo que calcula es determinístico, incluidas las cotas de la §5. |

> ### ⚠️ Datos restringidos
> Ninguna celda imprime filas individuales. Todo lo que sale es agregado. `nbstripout`
> limpia las salidas al commitear; las celdas de markdown no las toca el filtro, así que
> tampoco llevan datos reales.

> ### Por qué este notebook tenía que existir
> La cifra que se cita habitualmente —**37,6 alertas por 100 pacientes-día**— se midió en
> **sala general** y sobre otra población. Comparar el modelo propio contra ella sería
> compararlo contra otra cosa. La referencia tiene que salir de la misma cohorte, con la
> misma regla y el mismo motor.

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data import alertas, news2
from src.viz import estilo

estilo.aplicar()
pd.set_option("display.width", 120)

DATOS = RAIZ / "data" / "processed" / "mimic-iv-ed-demo-2.2"
INTERIM = RAIZ / "data" / "interim"
FIGURAS = INTERIM / "figuras"
FIGURAS.mkdir(parents=True, exist_ok=True)

obs = pd.read_parquet(DATOS / "observaciones_particionadas.parquet")
epi = pd.read_parquet(DATOS / "episodios.parquet")
puntuables = obs[obs.puntuable].copy()

print(f"tomas: {len(obs)}   puntuables: {len(puntuables)}   episodios: {len(epi)}")

## 1 · Qué cuenta como una alerta

**Qué espero encontrar antes de correr la celda:** que la regla real del sistema alerte
**bastante más** que un umbral de `score >= 5`, porque incluye el *rojo aislado*.

La regla que dispara una alerta en producción está en el trigger `emit_news2_alert` de la
migración `20260521000100_news2_engine.sql`:

```sql
IF NEW.risk_level IN ('Medio Bajo', 'Medio', 'Alto') THEN
```

es decir **cualquier nivel que no sea Bajo**. Eso incluye el caso en que un solo parámetro
puntúa 3 aunque el total sea bajo: un paciente con frecuencia respiratoria de 6 necesita
que alguien lo mire ya, sume lo que sume el resto. Es una decisión clínica de NEWS2.

**Por qué importa tanto:** si la referencia se midiera con `score >= 5`, estaría
describiendo un NEWS2 que este sistema **no implementa**, y toda la comparativa de
`SCRUM-58` quedaría anclada a un número equivocado. La regla vive en `src/data/alertas.py`
y hay una prueba que **lee la migración** y verifica que las dos no se hayan separado.

In [ ]:
marcadas = alertas.marcar(obs)
p = marcadas[marcadas.puntuable]

regla_sistema = p["alerta"]
regla_score5 = p["news2_score"] >= 5

comparacion = pd.DataFrame({
    "tomas que alertan": [int(regla_sistema.sum()), int(regla_score5.sum())],
    "% de las puntuables": [
        round(100 * regla_sistema.mean(), 1),
        round(100 * regla_score5.mean(), 1),
    ],
}, index=["regla real (riesgo != Bajo)", "regla equivocada (score >= 5)"])

print(comparacion.to_string())
print()
print(f"tomas que alertan SOLO por rojo aislado: {int((regla_sistema & ~regla_score5).sum())}")

In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 3.2))
valores = comparacion["% de las puntuables"]
barras = ax.barh(["score >= 5\n(equivocada)", "riesgo != Bajo\n(la del sistema)"],
                 [valores.iloc[1], valores.iloc[0]],
                 color=estilo.TERRACOTA, **estilo.BORDE_OBLIGATORIO)
for barra, valor, n in zip(barras, [valores.iloc[1], valores.iloc[0]],
                           [int(regla_score5.sum()), int(regla_sistema.sum())]):
    ax.annotate(f"{valor:.1f} %   ({n} tomas)",
                xy=(valor, barra.get_y() + barra.get_height() / 2),
                xytext=(6, 0), textcoords="offset points",
                va="center", fontsize=9.5, color=estilo.TINTA)
ax.set_xlabel("% de las tomas puntuables que disparan una alerta")
ax.set_title("Elegir mal el umbral cambia la referencia por más del doble",
             loc="left", color=estilo.TINTA)
ax.margins(x=0.22)
ax.grid(axis="y", visible=False)
fig.savefig(FIGURAS / "06_dos_reglas.png")
plt.show()

In [ ]:
desglose = alertas.desglose_rojo_aislado(obs)
print("Qué parámetro dispara el rojo aislado:")
print(desglose.to_string())

fig, ax = plt.subplots(figsize=(8, 3))
orden = desglose.sort_values()
barras = ax.barh([i.replace("_", " ") for i in orden.index], orden.values,
                 color=estilo.TERRACOTA, **estilo.BORDE_OBLIGATORIO)
for barra, valor in zip(barras, orden.values):
    ax.annotate(str(valor), xy=(valor, barra.get_y() + barra.get_height() / 2),
                xytext=(5, 0), textcoords="offset points",
                va="center", fontsize=9, color=estilo.TINTA)
ax.set_xlabel("Tomas con ese parámetro en 3 puntos")
ax.set_title("El rojo aislado lo dispara sobre todo la temperatura",
             loc="left", color=estilo.TINTA)
ax.margins(x=0.16)
ax.grid(axis="y", visible=False)
fig.savefig(FIGURAS / "06_rojo_aislado.png")
plt.show()

**Lo que se ve, y el problema que abre:** la temperatura es el parámetro que más dispara
el rojo aislado — y es **el que falta en el 44,2 % de las tomas** (notebook 05). O sea que
una parte importante de las alertas del sistema depende justo del dato peor registrado en
guardia. No invalida la referencia, pero es una fragilidad que hay que declarar: en la
base completa esa proporción puede moverse.

## 2 · Validar el motor antes de creerle el número

**Qué espero encontrar:** que los 21 casos clínicos pasen. Si el motor estuviera mal, la
tasa de referencia sería un número preciso de algo equivocado, que es peor que no tenerlo.

La implementación de Python (`src/data/news2.py`) y la de SQL (la migración) tienen que
coincidir: en producción calcula la base (ADR-001), y acá calcula Python. Las dos están
cubiertas por la misma batería de casos.

In [ ]:
import subprocess

resultado = subprocess.run(
    [sys.executable, str(RAIZ / "tests" / "test_news2.py")],
    capture_output=True, text=True,
)
print(resultado.stdout.strip() or resultado.stderr.strip())

resultado_alertas = subprocess.run(
    [sys.executable, str(RAIZ / "tests" / "test_alertas.py")],
    capture_output=True, text=True,
)
print(resultado_alertas.stdout.strip() or resultado_alertas.stderr.strip())

**Lo que NO está validado todavía, y conviene decirlo:** los 21 casos son de construcción
propia, derivados de la especificación del Royal College. **No hay validación externa**:
nadie comparó este motor contra puntajes NEWS publicados por un tercero.

Hay un candidato concreto para cerrarlo — el dataset del estudio **TRIAGE** en Zenodo
(`zenodo.org/records/4963759`, CC0, 1.303 pacientes **con NEWS ya calculado**). Una sola
toma por paciente, así que no sirve para la LSTM, pero sí serviría para verificar que este
motor reproduce puntajes de terceros. Queda anotado como trabajo futuro.

## 3 · La tasa de referencia, en tres denominadores

**Qué espero encontrar:** que los tres números cuenten historias distintas y que el de
*por paciente-día* salga muy por encima del 37,6 de la literatura.

Los tres, y para qué sirve cada uno:

- **Por toma** — el único que **no depende de cada cuánto se mide**. Es el que corresponde
  usar en `SCRUM-58`, porque el modelo puntúa exactamente las mismas tomas que NEWS2 y la
  frecuencia se cancela de los dos lados.
- **Por episodio** — responde "¿a qué fracción de pacientes el sistema le grita alguna
  vez?", que es la pregunta que hace un jefe de guardia.
- **Por 100 pacientes-día** — el único comparable con la literatura, y **el más engañoso
  de los tres**. Ver la §4.

In [ ]:
t = alertas.tasa(obs, epi)

tabla = pd.DataFrame([
    {"denominador": "por toma puntuable",
     "numerador": t.tomas_con_alerta, "denominador_n": t.tomas_puntuables,
     "tasa": f"{t.por_toma:.1%}"},
    {"denominador": "por episodio",
     "numerador": t.episodios_con_alerta, "denominador_n": t.episodios_puntuables,
     "tasa": f"{t.por_episodio:.1%}"},
    {"denominador": "por 100 pacientes-día",
     "numerador": t.tomas_con_alerta, "denominador_n": round(t.pacientes_dia, 1),
     "tasa": f"{t.por_100_pacientes_dia:.1f}"},
]).set_index("denominador")

print(tabla.to_string())
print()
print(f"referencia de la literatura (SALA general, otra población): "
      f"{alertas.REFERENCIA_LITERATURA_POR_100_PACIENTE_DIA} por 100 pacientes-día")
print(f"episodios sin ninguna toma puntuable: {t.episodios_sin_toma_puntuable} de {t.episodios_totales}")

## 4 · Por qué NO se puede decir "NEWS2 alerta el doble en guardia"

**Qué espero encontrar:** que la frecuencia de medición en guardia sea varias veces la de
una sala, y que eso explique buena parte de la diferencia contra el 37,6.

Es la trampa de este notebook y hay que desactivarla antes de que alguien la use mal.
La tasa por paciente-día **depende de cada cuánto se mide**, no sólo de cuánto alerta
NEWS2. En una sala se controla cada 6 o 12 h; en guardia, mucho más seguido. Si se mide
tres veces más, hay tres veces más oportunidades de alertar aunque el paciente y la escala
sean idénticos.

Decir "en guardia NEWS2 alerta el doble que en sala" a partir de estos dos números sería
**comparar dos cosas distintas y presentarlas como una**.

In [ ]:
lit = alertas.REFERENCIA_LITERATURA_POR_100_PACIENTE_DIA
print(f"tasa propia, por 100 pacientes-día : {t.por_100_pacientes_dia:.1f}")
print(f"literatura (sala)                  : {lit}")
print(f"cociente bruto                     : {t.por_100_pacientes_dia / lit:.1f}x")
print()
print(f"frecuencia de medición acá : {t.tomas_por_paciente_dia:.1f} tomas puntuables por paciente-día")
print(f"una sala con control c/6 h : 4.0 por paciente-día")
print(f"una sala con control c/12 h: 2.0 por paciente-día")
print()
factor = t.tomas_por_paciente_dia / 4.0
print(f"sólo por medir más seguido, el cociente ya se infla ~{factor:.1f}x")
print(f"cociente ajustado por frecuencia: {(t.por_100_pacientes_dia / lit) / factor:.2f}x")
print()
print("Es una corrección GRUESA, no una estandarización: la frecuencia de la sala")
print("de referencia es un supuesto, no un dato de esa publicación. Sirve para mostrar")
print("que la diferencia se explica en buena parte por el diseño de la medición.")

## 5 · La referencia es un PISO: cuánto la mueve la imputación

**Qué espero encontrar:** que el techo esté bastante por encima del 11,2 %, porque los dos
parámetros imputados son los que más pesan en NEWS2.

MIMIC-IV-ED no registra consciencia ni oxígeno suplementario **en ninguna fila**, así que
el ADR-007 los imputa al valor de menor riesgo: "A" y aire ambiente. El score resultante
nunca sobreestima. La referencia de la §3 es por lo tanto **un piso**, no una estimación.

Acá se acota cuánto podría subir. No se sortea nada: son **cotas deterministas**. Como la
tasa crece de forma monótona con los puntos agregados, basta preguntar cuántas tomas que
hoy no alertan cruzarían el umbral si recibieran los puntos que la imputación les quitó.

In [ ]:
s = p["news2_score"].to_numpy()
rojo = p["news2_rojo_aislado"].to_numpy()
alerta_base = p["alerta"].to_numpy()
n = len(s)

filas = []
for extra, nombre in [(0, "como está (todo imputado al mejor caso)"),
                      (2, "si TODAS tuvieran oxígeno suplementario (+2)"),
                      (3, "si TODAS tuvieran confusión nueva (+3)"),
                      (5, "si TODAS tuvieran ambos (+5)")]:
    cruzan = (~alerta_base) & ((s + extra) >= 5)
    tasa_nueva = (alerta_base | cruzan).mean()
    filas.append({"escenario": nombre,
                  "tomas que cruzan": int(cruzan.sum()),
                  "tasa por toma": f"{tasa_nueva:.1%}"})

cotas = pd.DataFrame(filas).set_index("escenario")
print(cotas.to_string())
print()
cruzan_o2 = int(((~alerta_base) & ((s + 2) >= 5)).sum())
print(f"Sólo {cruzan_o2} tomas están a 2 puntos o menos del umbral. Por eso la cota")
print(f"superior del escenario del oxígeno se satura: aunque el 100 % de las tomas")
print(f"tuviera oxígeno, no pueden cruzar más de {cruzan_o2}.")

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 3.4))
etiquetas = ["como está\n(el piso)", "+2 oxígeno", "+3 confusión", "+5 ambos"]
tasas = [float(x.strip("%")) for x in cotas["tasa por toma"]]

barras = ax.bar(etiquetas, tasas, color=estilo.rampa(4), **estilo.BORDE_OBLIGATORIO)
for barra, valor in zip(barras, tasas):
    ax.annotate(f"{valor:.1f} %",
                xy=(barra.get_x() + barra.get_width() / 2, barra.get_height()),
                xytext=(0, 4), textcoords="offset points",
                ha="center", fontsize=9.5, color=estilo.TINTA)

ax.axhline(tasas[0], color=estilo.OLIVA, linewidth=2)
ax.set_xlim(-0.6, 3.9)
ax.annotate("el piso que se\nreporta en §3", xy=(3.55, tasas[0]), ha="left", va="center",
            fontsize=8.5, color=estilo.OLIVA, linespacing=1.4)

ax.set_ylabel("% de tomas que alertan")
ax.set_title("La referencia es un piso: el techo de los escenarios está muy por encima",
             loc="left", color=estilo.TINTA)
ax.margins(y=0.22)
ax.grid(axis="x", visible=False)
fig.savefig(FIGURAS / "06_cotas_imputacion.png")
plt.show()

**El escenario del +5 llega al 100 % y no es un error:** ningún score de las tomas que hoy
no alertan supera los 4 puntos, así que sumarle 5 las cruza a todas. Es absurdo como
escenario clínico —nadie tiene confusión nueva *y* oxígeno en el 100 % de los controles—
pero dice algo real: **la imputación está haciendo un trabajo enorme**. Todo el resultado
descansa en que el ADR-007 sea conservador.

**Y acá está lo que hace defendible al proyecto:** el sesgo va en contra de la hipótesis.
Si la tasa real de NEWS2 es más alta que 11,2 %, entonces superarlo es *más fácil* de lo
que el número sugiere. Reportar el piso significa ponerse la vara **más alta**, no más
baja. Lo contrario —usar el techo— sería inflar el rival para ganarle.

## 6 · Lo que este notebook NO puede medir

Hay que separar dos cosas que suelen confundirse, porque una es usable sobre el demo y la
otra no:

| | ¿Necesita la etiqueta? | ¿Interpretable sobre el demo? |
|---|---|---|
| **Tasa de alertas** de NEWS2 | **No.** Sale del score y de la regla. | **Sí**, con la salvedad del tamaño de muestra. |
| **Sensibilidad** de NEWS2 | **Sí.** Hay que saber quién se deterioró. | **No.** La etiqueta colapsa (notebook 05, §4). |

Por eso este notebook entrega la **tasa** y no entrega la **sensibilidad**. Y por eso
`SCRUM-58` —"comparativa a sensibilidad igualada"— **no se puede cerrar con el demo**: para
igualar sensibilidades hay que poder medirlas, y sobre esta cohorte la etiqueta no
discrimina. Necesita la base completa (`SCRUM-48`).

Es una limitación del dato, no del método: el código de esta comparativa se escribe igual y
se corre apuntando a otra carpeta.

In [ ]:
# Salida para SCRUM-58. Agregados únicamente.
salida = pd.DataFrame([{
    "regla": " | ".join(alertas.RIESGOS_QUE_ALERTAN),
    "cohorte": "mimic-iv-ed-demo-2.2",
    "tomas_puntuables": t.tomas_puntuables,
    "tomas_con_alerta": t.tomas_con_alerta,
    "tasa_por_toma": round(t.por_toma, 4),
    "episodios_puntuables": t.episodios_puntuables,
    "episodios_con_alerta": t.episodios_con_alerta,
    "tasa_por_episodio": round(t.por_episodio, 4),
    "pacientes_dia": round(t.pacientes_dia, 2),
    "tasa_por_100_pacientes_dia": round(t.por_100_pacientes_dia, 2),
    "tomas_por_paciente_dia": round(t.tomas_por_paciente_dia, 2),
    "techo_si_todas_con_oxigeno": round(float(tasas[1]) / 100, 4),
    "techo_si_todas_con_confusion": round(float(tasas[2]) / 100, 4),
    "es_piso_por_imputacion_adr007": True,
}])

salida.to_parquet(INTERIM / "tasa_alertas_referencia.parquet", index=False)
print(f"guardado: {INTERIM / 'tasa_alertas_referencia.parquet'}")
print(salida.T.to_string(header=False))

## Conclusiones

**La referencia, para citar en la tesis y en `SCRUM-58`:**

> Sobre la cohorte MIMIC-IV-ED demo v2.2, aplicando la misma regla que el sistema
> despliega (`risk_level != 'Bajo'`), NEWS2 alerta en el **11,2 % de las tomas
> puntuables** (62 de 553) y en el **13,3 % de los episodios** (26 de 196). Es un **piso**:
> los dos parámetros que MIMIC no registra se imputan al valor de menor riesgo (ADR-007), y
> el techo del escenario con oxígeno suplementario es 20,4 %.

**Qué se concluyó**

1. **La regla importa más que el umbral.** La regla real alerta en el 11,2 % de las tomas;
   `score >= 5` daría 5,2 %. **Más del doble.** 33 tomas alertan sólo por rojo aislado.
2. **El rojo aislado lo dispara sobre todo la temperatura** (22 de 60), que es justo el
   parámetro que falta en el 44,2 % de las tomas. Una parte de las alertas depende del dato
   peor registrado en guardia.
3. **No se puede afirmar que NEWS2 alerte "el doble" en guardia.** La tasa por paciente-día
   sale ~2,2× la de la literatura, pero acá se mide ~7,4 veces por paciente-día contra 2–4
   en una sala. Buena parte de la diferencia es **diseño de medición**, no comportamiento
   de la escala.
4. **La referencia es un piso y el piso es la vara más exigente.** Si la tasa real es más
   alta, superarla es más fácil. Reportar el piso juega en contra de la hipótesis del
   proyecto, y por eso es defendible.
5. **El motor pasa sus 21 casos clínicos, pero no tiene validación externa.** Candidato
   para cerrarlo: el dataset TRIAGE en Zenodo (CC0, 1.303 pacientes con NEWS calculado).

**Qué se guardó**

- `data/interim/tasa_alertas_referencia.parquet` — la referencia completa, lista para `SCRUM-58`.
- `data/interim/figuras/06_*.png` — tres figuras.

**Qué queda abierto**

- **`SCRUM-58` no se puede cerrar con el demo.** Igualar sensibilidades exige poder medirlas
  y la etiqueta no discrimina sobre esta cohorte. Depende de `SCRUM-48`.
- **Re-correr este notebook sobre la base completa** en cuanto llegue la credencial. No
  cambia una línea: cambia `--origen`.
- **Validación externa del motor NEWS2** contra el dataset TRIAGE.